<a href="https://colab.research.google.com/github/JordanDCunha/Introduction-to-Machine-Learning-with-Python/blob/main/Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6. Algorithm Chains and Pipelines

For many machine learning algorithms, the representation of the data is crucial.
As discussed in Chapter 4, this includes:
- Scaling numerical features
- Encoding categorical variables
- Manually combining features
- Learning new features using unsupervised learning (Chapter 3)

As a result, most real-world machine learning workflows involve **multiple processing steps**, not just a single model.

These steps often include:
- Preprocessing (scaling, normalization, encoding)
- Feature extraction or transformation
- A final supervised learning model

Managing these steps manually can be error-prone and verbose.
To address this, scikit-learn provides the **Pipeline** class, which allows us to:
- Chain multiple transformations and a final estimator
- Treat the entire chain as a single model
- Safely combine preprocessing with cross-validation and grid search

In this chapter, we will:
- Introduce the `Pipeline` class
- Show how to combine preprocessing and models
- Use `Pipeline` together with `GridSearchCV` to tune parameters of *all* steps at once

## Motivation Example: Scaling + SVM

Kernel SVMs are highly sensitive to feature scales.
On the breast cancer dataset, performance improves significantly when features are scaled using `MinMaxScaler`.

Below is the **manual approach**, where preprocessing and modeling are handled step by step.


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# load and split the data
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0
)

# compute minimum and maximum on the training data
scaler = MinMaxScaler().fit(X_train)

# rescale the training data
X_train_scaled = scaler.transform(X_train)

# train SVM on scaled training data
svm = SVC()
svm.fit(X_train_scaled, y_train)

# scale the test data and evaluate
X_test_scaled = scaler.transform(X_test)
print("Test score: {:.2f}".format(svm.score(X_test_scaled, y_test)))


# 6.1 Parameter Selection with Preprocessing

When tuning model parameters using `GridSearchCV`, preprocessing must be handled carefully.
A common mistake is to apply preprocessing **before** cross-validation, which leads to **data leakage**.

## Naive Approach (Incorrect)

A naive approach is to:
- Scale the full training dataset first
- Run `GridSearchCV` on the already-scaled data

This seems reasonable but is **fundamentally flawed**.

### Why This Is a Problem
- Scaling computes statistics (min, max, mean, variance) from data
- When scaling is done **before** cross-validation:
  - Information from the validation folds leaks into training
  - The model indirectly “sees” data it should not have access to
- This leads to:
  - Overly optimistic cross-validation scores
  - Potentially selecting suboptimal hyperparameters

In real-world usage:
- New (test) data is **not** used when computing preprocessing statistics
- Therefore, cross-validation should simulate this exact scenario

## Correct Principle

**Any preprocessing step that learns from the data must be fit only on the training portion of each split.**

That means:
- Dataset splitting must happen first
- Preprocessing must occur *inside* the cross-validation loop

To enforce this correctly in scikit-learn, we use the **Pipeline** class.

## Pipeline Key Advantages
- Chains preprocessing and modeling into one estimator
- Ensures preprocessing is fit only on training folds
- Prevents data leakage
- Works seamlessly with `GridSearchCV` and `cross_val_score`

The Pipeline behaves like a standard estimator:
- `.fit()`
- `.predict()`
- `.score()`

This makes it the **recommended and safe** way to perform parameter tuning with preprocessing.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# ⚠️ For illustration only — DO NOT use this in practice
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(SVC(), param_grid=param_grid, cv=5)
grid.fit(X_train_scaled, y_train)

print("Best cross-validation accuracy: {:.2f}".format(grid.best_score_))
print("Best parameters:", grid.best_params_)
print("Test set accuracy: {:.2f}".format(grid.score(X_test_scaled, y_test)))


In [ ]:
import mglearn

# Visualization showing data leakage during cross-validation
mglearn.plots.plot_improper_processing()


# 6.2 Building Pipelines

The `Pipeline` class allows us to express a complete machine learning workflow—
including preprocessing and modeling—as a single estimator.

In this section, we build a pipeline that:
1. Scales the data using `MinMaxScaler`
2. Trains a Support Vector Machine (`SVC`)

A pipeline is created by providing a list of *steps*.
Each step is defined as a tuple:
- The first element is a **name** (any string)
- The second element is an **estimator instance**

The steps are executed sequentially in the order they are given.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

pipe = Pipeline([
    ("scaler", MinMaxScaler()),
    ("svm", SVC())
])


The pipeline behaves like any other scikit-learn estimator.

Calling `fit` on the pipeline performs the following steps internally:
1. Fit the scaler on the training data
2. Transform the training data using the scaler
3. Fit the SVM using the scaled data


In [ ]:
pipe.fit(X_train, y_train)


To evaluate the model on the test set, we simply call `score` on the pipeline.

Internally, this will:
1. Transform the test data using the already-fitted scaler
2. Compute the score using the SVM on the scaled test data

This produces the same result as manually scaling the data,
but with significantly less code.


In [ ]:
print("Test score: {:.2f}".format(pipe.score(X_test, y_test)))


While pipelines reduce boilerplate code, their real power is that they allow
the entire workflow to be treated as a single estimator.

This means we can now safely use the pipeline with:
- `cross_val_score`
- `GridSearchCV`

All preprocessing steps will be executed **inside** the cross-validation loop,
preventing data leakage and ensuring correct model evaluation.


# 6.3 Using Pipelines in Grid Searches

Using a pipeline with `GridSearchCV` works the same way as with any estimator,
with one important difference in how parameters are specified.

Key ideas:
- We define a parameter grid as usual
- Each parameter must be associated with a **specific pipeline step**
- Parameters are referenced using the syntax:  
  `step_name__parameter_name`
- Double underscores (`__`) separate the step name from the parameter

In this example:
- The pipeline step is named `"svm"`
- The parameters `C` and `gamma` belong to `SVC`
- Therefore, the parameters are written as `svm__C` and `svm__gamma`


In [ ]:
# 6.3 Using Pipelines in Grid Searches

Using a pipeline with `GridSearchCV` works the same way as with any estimator,
with one important difference in how parameters are specified.

Key ideas:
- We define a parameter grid as usual
- Each parameter must be associated with a **specific pipeline step**
- Parameters are referenced using the syntax:
  `step_name__parameter_name`
- Double underscores (`__`) separate the step name from the parameter

In this example:
- The pipeline step is named `"svm"`
- The parameters `C` and `gamma` belong to `SVC`
- Therefore, the parameters are written as `svm__C` and `svm__gamma`


param_grid = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}


In [ ]:
We can now construct a `GridSearchCV` object using the pipeline and parameter grid.

Important:
- For each cross-validation split:
  - The scaler is fit **only on the training fold**
  - The model is trained on the transformed training data
  - The test fold remains completely unseen
- This prevents **data leakage** during model selection


from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best cross-validation accuracy: {:.2f}".format(grid.best_score_))
print("Test set score: {:.2f}".format(grid.score(X_test, y_test)))
print("Best parameters: {}".format(grid.best_params_))


We now demonstrate how information leakage can severely distort results.

Setup:
- 100 samples
- 10,000 features
- Features and target are generated randomly
- There is **no real relationship** between X and y

Expected outcome:
- A good model should NOT be able to learn anything meaningful


In [ ]:
import numpy as np

rnd = np.random.RandomState(seed=0)
X = rnd.normal(size=(100, 10000))
y = rnd.normal(size=(100,))


Here, feature selection is applied **before** cross-validation.

Problem:
- Feature selection sees the entire dataset
- Features correlated with the target are selected by chance
- Information from the test folds leaks into training


In [ ]:
from sklearn.feature_selection import SelectPercentile, f_regression

select = SelectPercentile(score_func=f_regression, percentile=5).fit(X, y)
X_selected = select.transform(X)

print("X_selected.shape:", X_selected.shape)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge

print("Cross-validation accuracy (cv only on ridge): {:.2f}".format(
    np.mean(cross_val_score(Ridge(), X_selected, y, cv=5))
))


Now, feature selection is placed **inside** the pipeline.

Result:
- Feature selection is fit only on training folds
- No information from test folds is leaked
- Performance reflects the true difficulty of the task


In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("select", SelectPercentile(score_func=f_regression, percentile=5)),
    ("ridge", Ridge())
])

print("Cross-validation accuracy (pipeline): {:.2f}".format(
    np.mean(cross_val_score(pipe, X, y, cv=5))
))


Key lessons from this section:

- Always include preprocessing steps inside a pipeline
- Never perform feature selection or transformation before cross-validation
- Pipelines prevent information leakage automatically
- Data leakage can completely invalidate model evaluation
- Pipelines are essential for reliable grid search and model comparison


# 6.4 The General Pipeline Interface

Pipelines are not limited to preprocessing + classification.

Key ideas:
- A pipeline can chain **any number of estimators**
- Typical steps may include:
  - Feature extraction
  - Feature selection
  - Scaling
  - Classification, regression, or clustering
- Only requirement:
  - All steps **except the last** must implement `transform`
  - The last step must implement `fit`


# 6.4 The General Pipeline Interface

Pipelines are not limited to preprocessing + classification.

Key ideas:
- A pipeline can chain **any number of estimators**
- Typical steps may include:
  - Feature extraction
  - Feature selection
  - Scaling
  - Classification, regression, or clustering
- Only requirement:
  - All steps **except the last** must implement `transform`
  - The last step must implement `fit`


In [ ]:
Internally, `Pipeline.fit` works as follows:

- Start with the original input data
- For each step except the last:
  - Call `fit`
  - Then call `transform`
  - Pass the transformed data to the next step
- For the final step:
  - Only `fit` is called

Each step operates on the output of the previous step


def fit(self, X, y):
    X_transformed = X
    for name, estimator in self.steps[:-1]:
        # fit and transform all but the last step
        X_transformed = estimator.fit_transform(X_transformed, y)
    # fit the final step
    self.steps[-1][1].fit(X_transformed, y)
    return self


In [ ]:
During prediction:

- The data is transformed step by step
- All steps except the last apply `transform`
- The final step applies `predict`


def predict(self, X):
    X_transformed = X
    for step in self.steps[:-1]:
        # transform the data
        X_transformed = step[1].transform(X_transformed)
    # predict with the final estimator
    return self.steps[-1][1].predict(X_transformed)


The last step of a pipeline does NOT need to implement `predict`.

Examples:
- A pipeline with only transformers (e.g., scaler + PCA)
- Calling `transform` on the pipeline applies all transformations

Requirement:
- The final step must have `fit`
- If it has `transform`, the pipeline can expose `transform`


In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

# explicit naming
pipe_long = Pipeline([
    ("scaler", MinMaxScaler()),
    ("svm", SVC(C=100))
])

# automatic naming
pipe_short = make_pipeline(
    MinMaxScaler(),
    SVC(C=100)
)


In [ ]:
print("Pipeline steps:\n{}".format(pipe_short.steps))


Automatic step naming rules:
- Step names are lowercase class names
- If a class appears multiple times:
  - A number is appended (e.g., `standardscaler-1`)
- For complex pipelines:
  - Explicit names may be clearer and more semantic


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    StandardScaler()
)

print("Pipeline steps:\n{}".format(pipe.steps))


Often we want to inspect internal attributes of pipeline steps.

Examples:
- Coefficients of a linear model
- Components extracted by PCA

Best approach:
- Use the `named_steps` attribute
- This is a dictionary mapping step names to estimators


In [ ]:
# fit the pipeline to the cancer dataset
pipe.fit(cancer.data)

# access PCA components
components = pipe.named_steps["pca"].components_
print("components.shape:", components.shape)


Pipelines are especially useful when combined with grid search.

Common use case:
- Preprocessing + model selection
- Accessing attributes of the best model after tuning

Key concept:
- `GridSearchCV.best_estimator_` stores the fully trained pipeline


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)


In [ ]:
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10, 100]
}


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=4
)

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)


The best model found by grid search is stored in `best_estimator_`.

This object:
- Is a fully trained pipeline
- Contains all preprocessing and model steps


In [ ]:
print("Best estimator:\n{}".format(grid.best_estimator_))


To access a specific step inside the best pipeline:
- Use `best_estimator_.named_steps`
- Retrieve the estimator by its step name


In [ ]:
print("Logistic regression step:\n{}".format(
    grid.best_estimator_.named_steps["logisticregression"]
))


In [ ]:
print("Logistic regression coefficients:\n{}".format(
    grid.best_estimator_.named_steps["logisticregression"].coef_
))


Key takeaways from Section 6.4:

- Pipelines can chain any sequence of estimators
- Only the last step must implement `fit`
- `make_pipeline` simplifies pipeline creation
- `named_steps` allows easy access to internal estimators
- Pipelines integrate seamlessly with GridSearchCV
- Inspecting tuned models is straightforward and powerful


## 6.5 Grid-Searching Preprocessing Steps and Model Parameters

### Key Ideas
- Pipelines allow **preprocessing and modeling** to be treated as a single estimator.
- This lets us **tune preprocessing parameters** (like polynomial degree) using supervised performance.
- GridSearchCV can search **both preprocessing and model parameters at once**.
- All parameters must be prefixed with their **pipeline step name**.


### Example Setup: Polynomial Features + Ridge Regression
- Dataset: Boston housing
- Pipeline steps:
  - Standardize features
  - Generate polynomial features
  - Apply Ridge regression


In [ ]:
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

boston = load_boston()

X_train, X_test, y_train, y_test = train_test_split(
    boston.data, boston.target, random_state=0
)

pipe = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(),
    Ridge()
)


### Grid-Searching Preprocessing and Model Parameters
- Tune:
  - `degree` of PolynomialFeatures
  - `alpha` of Ridge regression
- Parameters are prefixed using:
  - `polynomialfeatures__degree`
  - `ridge__alpha`


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'polynomialfeatures__degree': [1, 2, 3],
    'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}


### Running GridSearchCV
- 5-fold cross-validation
- Parallelized using all available cores


In [ ]:
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)


### Visualizing Cross-Validation Results
- Heat map shows interaction between:
  - Polynomial degree
  - Ridge regularization strength
- Helps identify overfitting vs underfitting


In [ ]:
import matplotlib.pyplot as plt

plt.matshow(
    grid.cv_results_['mean_test_score'].reshape(3, -1),
    vmin=0,
    cmap="viridis"
)
plt.xlabel("ridge__alpha")
plt.ylabel("polynomialfeatures__degree")
plt.xticks(
    range(len(param_grid['ridge__alpha'])),
    param_grid['ridge__alpha']
)
plt.yticks(
    range(len(param_grid['polynomialfeatures__degree'])),
    param_grid['polynomialfeatures__degree']
)
plt.colorbar()
plt.show()


### Best Parameters Found
- Polynomial degree of **2** performs best
- Moderate Ridge regularization works best


In [ ]:
print("Best parameters:", grid.best_params_)


### Test Set Performance


In [ ]:
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))


### Comparison: No Polynomial Features
- Remove PolynomialFeatures from the pipeline
- Grid-search only Ridge regression


In [ ]:
pipe = make_pipeline(StandardScaler(), Ridge())

param_grid = {
    'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Score without poly features: {:.2f}".format(
    grid.score(X_test, y_test)
))


### Takeaways
- Polynomial features significantly improve performance
- Degree 3 causes overfitting
- Grid-searching preprocessing + model parameters is powerful
- Beware: grid size grows **exponentially** with more parameters


## 6.6 Grid-Searching Which Model To Use

### Key Ideas
- Pipelines can be used to **search over different models**, not just parameters.
- GridSearchCV can choose:
  - Which classifier to use
  - Whether preprocessing is applied
- This creates a **much larger search space**, so it should be used carefully.


## 6.6 Grid-Searching Which Model To Use

### Key Ideas
- Pipelines can be used to **search over different models**, not just parameters.
- GridSearchCV can choose:
  - Which classifier to use
  - Whether preprocessing is applied
- This creates a **much larger search space**, so it should be used carefully.


### Example Scenario
- Compare:
  - Support Vector Classifier (SVC)
  - RandomForestClassifier
- Dataset: Breast cancer
- Observations:
  - SVC often benefits from feature scaling
  - Random Forests do **not** require preprocessing


In [ ]:
### Pipeline Design
- Two pipeline steps:
  1. Preprocessing
  2. Classifier
- Steps are **explicitly named** so they can be replaced in GridSearchCV


### Parameter Grid with Multiple Models
- Use a **list of dictionaries** for different model configurations
- Allows:
  - Different parameters per model
  - Skipping preprocessing by setting a step to `None`


In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid = [
    {
        'classifier': [SVC()],
        'preprocessing': [StandardScaler(), None],
        'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
        'classifier__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'classifier': [RandomForestClassifier(n_estimators=100)],
        'preprocessing': [None],
        'classifier__max_features': [1, 2, 3]
    }
]


### Running the Grid Search
- GridSearchCV automatically:
  - Chooses the best model
  - Selects preprocessing strategy
  - Tunes hyperparameters


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0
)

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best parameters:\n", grid.best_params_)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))


### Results
- Best model selected:
  - SVC with StandardScaler
- Best hyperparameters:
  - C = 10
  - gamma = 0.01
- Indicates:
  - Scaling was beneficial
  - SVC outperformed Random Forest on this dataset


## 6.6.1 Avoiding Redundant Computation

### Problem
- In large grid searches:
  - Same preprocessing steps are recomputed many times
- This is inefficient for:
  - PCA
  - NMF
  - Other expensive transformations


### Solution: Caching with Pipeline
- Use the `memory` parameter of Pipeline
- Stores fitted transformers on disk
- Prevents recomputation when reused


In [ ]:
pipe = Pipeline(
    [('preprocessing', StandardScaler()), ('classifier', SVC())],
    memory="cache_folder"
)


### Downsides of Caching
- Disk I/O overhead:
  - Serialization + reading/writing
- Only useful for **expensive transformations**
- May conflict with parallel execution (`n_jobs`)


### Advanced Alternative: dask-ml
- Avoids redundant computation
- Supports:
  - Parallel execution
  - Distributed computing
- Recommended for:
  - Large pipelines
  - Extensive hyperparameter searches


### Takeaways
- Pipelines + GridSearchCV can:
  - Choose the best model
  - Choose preprocessing
  - Tune parameters jointly
- Very powerful — but **computationally expensive**
- Use caching or dask-ml for large searches


## 6.7 Summary and Outlook

### What This Chapter Introduced
- The **Pipeline** class for chaining multiple ML steps
- Pipelines combine:
  - Feature extraction
  - Preprocessing
  - Model training
- Pipelines behave like any scikit-learn estimator:
  - `fit`
  - `predict`
  - `transform`

---

### Why Pipelines Matter
- Real-world ML workflows are **not single models**
- Pipelines:
  - Encapsulate all steps into one object
  - Enforce correct order of operations
  - Prevent common mistakes (e.g., forgetting preprocessing on test data)

---

### Pipelines and Model Evaluation
- Essential for:
  - **Cross-validation**
  - **Grid search**
- Ensure:
  - No information leakage
  - Fair evaluation of model performance
- All preprocessing happens **inside** the validation loop

---

### Code Quality Benefits
- More concise and readable code
- Fewer bugs caused by manual chaining
- Easier experimentation with different workflows

---

### Practical Guidance
- Choosing preprocessing + model combinations:
  - Is partly trial and error
  - Is made easier with pipelines
- Avoid overengineering:
  - Each pipeline component should have a clear purpose
  - Remove unnecessary steps

---

### Big Picture
- This chapter completes the overview of:
  - Core scikit-learn tools
  - Model evaluation and selection techniques
- You now have the skills to:
  - Build
  - Evaluate
  - Tune machine learning systems properly

---

### What’s Next
- Next chapter focus:
  - **Text data**
- Text requires:
  - Specialized representations
  - Domain-specific preprocessing
